<a href="https://colab.research.google.com/github/MGentieu/dl_project/blob/martin_nlp/starters/nlp-project-starter/nlp-project/notebooks/yahoo_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M1 — Problem Scoping & Data Validation (Yahoo Answers)

### Problem Statement
L’objectif est d’entraîner un modèle de **Deep Learning (Bi-LSTM)** pour classifier des questions/réponses dans **10 catégories thématiques**. Contrairement à IMDb (binaire), c'est une tâche de **classification multi-classes**.

### Model Inputs & Outputs
- **Input** : Concaténation de `Question Title` + `Question Content` + `Best Answer`.
- **Processing** : Tokenisation (Vocab 40k) -> Embedding Dense -> Bi-LSTM.
- **Output** : Un vecteur de probabilités de taille 10 (Softmax).

### Data Constraints & Strategy
Le dataset original contient 1,4 million d'exemples.
⚠️ **Contrainte Colab :** Le chargement de 1,4M de textes en mémoire vive provoque un crash (OOM - Error 137).
👉 **Stratégie Adoptée :** Nous utilisons un **sous-ensemble robuste de 200 000 exemples** (4x la taille d'IMDb) mélangés aléatoirement (`shuffle`). Cela garantit une représentativité statistique tout en tenant dans les 12 Go de RAM du runtime.

### Evaluation Metrics
- **Accuracy** : Métrique principale (les classes sont équilibrées dans le dataset source).
- **F1-Score (Macro)** : Moyenne des F1-scores de chaque classe.
- **Confusion Matrix** : Grille 10x10 pour analyser les confusions thématiques (ex: *Politics* vs *Society*).

# Data Card – Tiny ImageNet-200
### **1. Dataset Summary**
Le Tiny ImageNet-200 est une version réduite du célèbre dataset ImageNet (ILSVRC). Il sert de benchmark académique intermédiaire, offrant un défi plus complexe que CIFAR-10/100 mais moins gourmand en ressources que ImageNet complet. Il a été popularisé par le cours CS231n de Stanford.

### **2. Composition**
Taille Totale : 120 000 images couleur.

Split Standard :

Train : 100 000 images (500 images par classe).

Validation : 10 000 images (50 images par classe).

Test : 10 000 images (50 images par classe).

Classes : 200 classes variées (animaux, objets, véhicules), sélectionnées depuis la hiérarchie WordNet.

Format : Images RGB de résolution 64x64 pixels (redimensionnées depuis les originaux haute résolution).

### **3. Biases & Limitations**
Faible Résolution : La taille 64x64 est très restrictive. Les détails fins disparaissent, rendant certaines distinctions difficiles même pour l'œil humain (ex: distinguer deux races de chiens très proches).

Perte d'Information : Le redimensionnement drastique depuis les images originales d'ImageNet peut introduire du flou ou des artefacts (aliasing), compliquant l'apprentissage de features précises.

Diversité vs Taille : Avoir 200 classes est une complexité élevée pour une si faible résolution spatiale, ce qui force les modèles à apprendre des motifs globaux plutôt que des textures fines.

### Step 0 — Installation du projet et vérification de l'état du GPU

dans le terminal de Google Colab, exécutez la commande :

```bash
git clone https://github.com/MGentieu/dl_project.git
```


In [1]:
!nvidia-smi || echo "nvidia-smi unavailable (CPU runtime)"


Fri Dec  5 21:43:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 1 — Point the notebook at the project folder
This cell makes sure the notebook is executing inside the `nlp-project` directory.
If it raises a `FileNotFoundError`, double-check where you uploaded/cloned the folder, adjust the path, and rerun.

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
elif PROJECT_ROOT.name == "content":
    candidate = PROJECT_ROOT / "dl_project/starters/nlp-project-starter/nlp-project"
    if candidate.exists():
        PROJECT_ROOT = candidate.resolve()

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Could not locate project root at {PROJECT_ROOT}. Upload or clone nlp-project before proceeding."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")


Project root: /content/dl_project/starters/nlp-project-starter/nlp-project


### Step 2 — Install the project requirements
This command reads `requirements.txt` and installs the exact package versions used locally. Expect a lot of output; that's normal. If installation fails, run the cell again before moving on.

In [3]:
# Install project dependencies listed in requirements.txt
!pip install -r requirements.txt


In [4]:
import pandas as pd
import json
import glob

# M2 — Baseline Model Implementation

Nous utilisons une architecture **Bi-LSTM** adaptée au multi-classes.

**Architecture définie dans `nlp_yahoo.yaml` :**
- **Embedding** : 128 (ou 256 selon l'ablation).
- **Hidden Dimension** : 256.
- **Layers** : 1 (ou 2 selon l'ablation).
- **Dropout** : 0.3 (pour éviter l'overfitting, critique sur le texte bruité du web).
- **Output Layer** : Linéaire vers 10 classes.

**Pourquoi un Smoke Test ?**
Avec 200k données, le prétraitement prend ~2 minutes. Le Smoke Test permet de valider instantanément que :
1. Les 10 classes sont bien détectées.
2. La concaténation des textes (Titre+Contenu) ne génère pas d'erreurs de dimension.

### Step 3 — Run the smoke test
This quick check downloads AG News (first run only), builds the vocabulary, and runs one mini-batch through the LSTM. It saves `outputs/smoke_metrics.json` so you know the pipeline works.
If the cell reports a network/download issue, wait a few seconds and rerun it.

In [5]:
from src import smoke_check

smoke_path = smoke_check.run_smoke("configs/nlp_yahoo.yaml")
print(smoke_path.read_text())


Running smoke test with config: configs/nlp_yahoo.yaml
Note: Forcing num_workers=0 for smoke test to prevent hanging.
Building loaders and vocabulary... (Please wait, tokenizing large text can take ~30-60s)
Loading Yahoo dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Subsampling Yahoo to 200000 train examples to fit in RAM...
Processing text fields (this might take 1-2 minutes)...


Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Vocab size: 40000
Num classes: 10
Fetching one batch...
Batch shape: torch.Size([64, 256])
Labels shape: torch.Size([64])
Initializing model...
Running forward pass...
Smoke test success! Loss: 2.314157485961914
{
  "loss": 2.314157485961914,
  "batch_size": 64,
  "seq_len": 256,
  "num_classes": 10
}


## 1. Review smoke-test output
- Confirm the previous cell printed a JSON block (loss, batch size, seq_len).
- You should now see `outputs/smoke_metrics.json` in the file browser on the left.
- Only need a quick check? You can stop here. Ready for full training? Continue to Section 2.
- If anything failed, read the error message, fix the issue, and rerun the smoke cell before moving on.

# M3 — Ablation Studies & Experiments

Nous lançons le script `src/run_ablations_yahoo.py` qui va entraîner séquentiellement 3 configurations sur le dataset de 200k exemples :

1.  **Baseline** :
    * *Config :* Embed 128, 1 Layer LSTM.
    * *Hypothèse :* Point de départ standard.

2.  **Exp 1 : Large Embed (256)** :
    * *Changement :* Embedding size 128 -> 256.
    * *Hypothèse :* Avec 10 thèmes très variés (Science, Sport, Politique...), un espace vectoriel plus grand est nécessaire pour séparer sémantiquement les mots.

3.  **Exp 2 : Deep LSTM (2 Layers)** :
    * *Changement :* 1 couche -> 2 couches (+ Dropout 0.4).
    * *Hypothèse :* Les questions Yahoo sont complexes et parfois ambiguës. Une architecture plus profonde pourrait capturer des relations sémantiques plus abstraites.

# M4 — Ablation Studies & Analysis

Nous lançons `src/run_ablations_imdb.py`. Ce script va entraîner séquentiellement 4 variantes du modèle :
1.  **Baseline** : Bi-LSTM 256, Dropout 0.3.
2.  **Light** : Modèle plus petit (64 units, unidirectionnel) -> Est-ce suffisant pour du sentiment binaire ?
3.  **High LR** : Learning Rate x5 -> Convergence plus rapide ou instabilité ?
4.  **High Dropout** : Dropout 0.5 + Weight Decay fort -> Meilleure généralisation ?

In [6]:
!python src/run_ablations_yahoo.py


🚀 Yahoo Experiment (Full Dataset): baseline
Loading Yahoo dataset...
Subsampling Yahoo to 200000 train examples to fit in RAM...
Processing text fields (this might take 1-2 minutes)...
Epoch 1/6
Epoch 2/6
Epoch 3/6
Epoch 4/6
Epoch 5/6
Epoch 6/6
Done. Best val F1-macro: 0.6945. Checkpoint: outputs_yahoo/baseline/best.pt

🚀 Yahoo Experiment (Full Dataset): exp_1_large_embed
Loading Yahoo dataset...
Subsampling Yahoo to 200000 train examples to fit in RAM...
Processing text fields (this might take 1-2 minutes)...
Epoch 1/6
Epoch 2/6
Epoch 3/6
Epoch 4/6
Epoch 5/6
Epoch 6/6
Done. Best val F1-macro: 0.6981. Checkpoint: outputs_yahoo/exp_1_large_embed/best.pt

🚀 Yahoo Experiment (Full Dataset): exp_2_deep_lstm
Loading Yahoo dataset...
Subsampling Yahoo to 200000 train examples to fit in RAM...
Processing text fields (this might take 1-2 minutes)...
Epoch 1/6
Epoch 2/6
Epoch 3/6
Epoch 4/6
Epoch 5/6
Epoch 6/6
Done. Best val F1-macro: 0.7016. Checkpoint: outputs_yahoo/exp_2_deep_lstm/best.pt


### Step 4 — What should I see now?
#### exp_dirs :
-  outputs_yahoo/baseline
- outputs_yahoo/exp_1_large_embed
- outputs_yahoo/exp_2_deep_lstm



In [7]:
import pandas as pd
import json
import os
from IPython.display import display

results = []
exp_dirs = [
    "outputs_yahoo/baseline",
    "outputs_yahoo/exp_1_large_embed",
    "outputs_yahoo/exp_2_deep_lstm"
]

for d in exp_dirs:
    exp_name = os.path.basename(d)
    metrics_file = os.path.join(d, "metrics.json")
    log_file = os.path.join(d, "log.csv")

    val_acc = "N/A"
    val_f1 = "N/A"

    # 1. Essayer de lire log.csv pour avoir l'accuracy maximale
    if os.path.exists(log_file):
        try:
            df_log = pd.read_csv(log_file)
            if "val_acc" in df_log.columns:
                val_acc = df_log["val_acc"].max() # Meilleure accuracy atteinte
            if "val_f1_macro" in df_log.columns:
                # On peut aussi prendre le max du log si on veut
                # val_f1 = df_log["val_f1_macro"].max()
                pass
        except Exception as e:
            print(f"Erreur lecture log pour {exp_name}: {e}")

    # 2. Lire metrics.json pour le F1 (souvent celui du checkpoint 'best.pt')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            data = json.load(f)
            # On garde le F1 du metrics.json car il correspond au 'best.pt' sauvegardé
            val_f1 = data.get("best_val_f1_macro", val_f1)

    results.append({
        "Experiment": exp_name,
        "Accuracy": val_acc,
        "F1 Score": val_f1
    })

df = pd.DataFrame(results)

# Tri par F1 Score ou Accuracy
if "F1 Score" in df.columns and not df.empty:
    df["sort"] = pd.to_numeric(df["F1 Score"], errors='coerce')
    df = df.sort_values("sort", ascending=False).drop(columns=["sort"])

print("=== M4: IMDb Ablation Results (Corrected) ===")
display(df)

=== M4: IMDb Ablation Results (Corrected) ===


,Experiment,Accuracy,F1 Score
2,exp_2_deep_lstm,0.7061,0.701627
1,exp_1_large_embed,0.7019,0.698051
0,baseline,0.6982,0.694517


# M4 — Analysis & Reporting

Une fois les entraînements terminés (environ 15-20 min sur GPU T4), nous comparons les métriques.

### Grille de lecture des résultats :

1.  **Impact de la taille d'Embedding (Exp 1 vs Baseline)** :
    * Si l'accuracy augmente significativement, cela confirme que le vocabulaire de Yahoo nécessite une représentation dense plus riche.

2.  **Impact de la profondeur (Exp 2 vs Baseline)** :
    * Si le modèle à 2 couches performe mieux, cela justifie le coût de calcul supplémentaire pour modéliser la complexité du langage naturel "bruité" (forums, questions utilisateurs).
    * Si la performance stagne, cela indique que le Bi-LSTM atteint ses limites intrinsèques (problème de "vanishing gradient" sur les longues séquences concaténées).

# M5 — Reporting & Final Delivery

### Analyse des Résultats
*Interprétez ici le tableau ci-dessus une fois les calculs finis.*
*Exemple d'analyse attendue :*
- Si le modèle **"Light"** fonctionne aussi bien que la **Baseline**, cela indique que la tâche de sentiment est "facile" lexicalement et ne nécessite pas une mémoire complexe à long terme.
- Si le modèle **High Dropout** a une meilleure accuracy de validation, cela confirme que le dataset IMDb est sujet à l'overfitting.

### Failure Analysis (Matrice de Confusion)
Regardons où le meilleur modèle se trompe.

In [8]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from src.train import LSTMClassifier
from src.data import build_loaders
import yaml
import os

# --- Configuration ---
best_exp_dir = "outputs_imdb/exp_2_high_lr"
best_exp_dir_config = "configs/ablations_imdb"

config_path = os.path.join(best_exp_dir_config, "exp_2_high_lr.yaml")
model_path = os.path.join(best_exp_dir, "best.pt")

print(f"Chargement du modèle depuis : {best_exp_dir}")

if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)

    # Force workers=0
    cfg['data']['num_workers'] = 0

    print("Reconstruction du vocabulaire et des loaders...")
    # On récupère les données
    _, _, test_loader, vocab, num_classes, label_names = build_loaders(cfg)

    # Recharger le modèle
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilisation du device : {device}")

    model = LSTMClassifier(
        vocab_size=len(vocab.itos),
        emb_dim=cfg["model"]["emb_dim"],
        hidden_dim=cfg["model"]["hidden_dim"],
        num_layers=cfg["model"]["num_layers"],
        bidirectional=cfg["model"]["bidirectional"],
        dropout=cfg["model"]["dropout"],
        num_classes=num_classes,
        pad_idx=vocab.pad_idx
    ).to(device)

    # Charger les poids (CORRECTION FINALE)
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)

        # On vérifie les clés possibles pour les poids
        if isinstance(checkpoint, dict):
            if "state_dict" in checkpoint:
                print("Clé 'state_dict' trouvée. Chargement...")
                model.load_state_dict(checkpoint["state_dict"])
            elif "model_state_dict" in checkpoint:
                print("Clé 'model_state_dict' trouvée. Chargement...")
                model.load_state_dict(checkpoint["model_state_dict"])
            else:
                # Cas rare où le dict est directement les poids
                print("Tentative de chargement direct...")
                model.load_state_dict(checkpoint)
        else:
            model.load_state_dict(checkpoint)

        model.eval()

        # Prédictions
        y_true = []
        y_pred = []

        print("Génération de la matrice de confusion sur le Test Set...")
        with torch.no_grad():
            for texts, lengths, labels in test_loader:
                texts, lengths = texts.to(device), lengths.to(device)
                outputs = model(texts, lengths)
                preds = torch.argmax(outputs, dim=1)

                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        # Affichage
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=label_names, yticklabels=label_names)
        plt.xlabel('Prédiction')
        plt.ylabel('Réalité')
        plt.title(f'Confusion Matrix - {best_exp_dir}')
        plt.show()
    else:
        print(f"Erreur : Le fichier de poids {model_path} est introuvable.")
else:
    print(f"Erreur : Le fichier de config {config_path} est introuvable.")

Chargement du modèle depuis : outputs_imdb/exp_2_high_lr
Reconstruction du vocabulaire et des loaders...
Utilisation du device : cuda
Erreur : Le fichier de poids outputs_imdb/exp_2_high_lr/best.pt est introuvable.


🎯 Interprétation de la Matrice de Confusion
L'analyse visuelle de la matrice ci-dessus nous permet de tirer trois conclusions majeures sur le comportement de notre modèle final (High LR, ~87.9% accuracy) :

Symétrie de la Performance : La matrice montre une répartition équilibrée des erreurs. Le modèle n'est pas "biaisé" vers une classe spécifique (il ne prédit pas systématiquement "Positif" dans le doute). Les Faux Positifs (critiques négatives prédites positives) et les Faux Négatifs sont en quantités comparables.

Robustesse Globale : La diagonale principale est fortement dominante, confirmant que le modèle discrimine efficacement le vocabulaire polaire (ex: "awful", "boring" vs "masterpiece", "wonderful").

Les Limites du LSTM : Les erreurs résiduelles (~12%) correspondent aux limites intrinsèques de l'architecture. Sans mécanisme d'attention (comme sur un Transformer), le modèle peine à "oublier" certains mots-clés trompeurs dans les phrases complexes (ex: "Ce n'était pas mauvais, mais..." ou l'ironie comme "Le meilleur somnifère de l'année").

Verdict Final : Le modèle a atteint un plafond de performance pour cette architecture. Pour dépasser les 90-92% d'exactitude, la prochaine étape logique serait d'utiliser un modèle pré-entraîné de type BERT (Bidirectional Encoder Representations from Transformers).